# Week 5 Day 5 — Model Stacking + Ensembling

## What I built:
- Simple averaging — RF + XGB + LGB equally weighted
- Weighted averaging — more weight to better models
- Blend ratio tuning — tested 5 LGB/XGB ratios
- Stacking — XGB + LGB as base, Ridge as meta-learner

## Results summary:
Simple average (RF+XGB+LGB)  : 0.12877  ← RF hurts ensemble
Weighted (LGB50+XGB35+RF15)  : 0.12679
LGB alone                    : 0.12603
Blend LGB60+XGB40            : 0.12587  ← best Kaggle 🏆
Stacking CV                  : 0.1256   ← best CV
Stacking Kaggle              : 0.12666  ← overfit on small data

## Key learnings:

**Blending vs Stacking:**
Blending  → manual weights, fast, works well on small data
Stacking  → meta-model learns weights, better on large data
            XGB+LGB → Ridge meta-learner
            cv=5 prevents data leakage in stacking

**Why RF hurt the ensemble:**
RF Kaggle score = 0.14864 — too weak
Even 15% weight pulled ensemble down
Only blend models with similar strength

**Blend ratio sweet spot:**
LGB 60% + XGB 40% = 0.12587
Very flat curve 50-70% LGB — both models contribute equally well

**Rule:**
Large dataset (>10k) → stacking wins
Small dataset (<5k)  → blending safer

# Simple averaging:

In [1]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import KFold, cross_val_predict
import warnings
warnings.filterwarnings('ignore')

# load data
train = pd.read_csv('house_prices_train.csv')
test  = pd.read_csv('house_prices_test.csv')

def engineer_features(df):
    df = df.copy()
    df['TotalSF']     = df['TotalBsmtSF'] + df['1stFlrSF'] + df['2ndFlrSF']
    df['TotalBath']   = (df['FullBath'] + 0.5*df['HalfBath'] +
                         df['BsmtFullBath'] + 0.5*df['BsmtHalfBath'])
    df['HouseAge']    = df['YrSold'] - df['YearBuilt']
    df['RemodAge']    = df['YrSold'] - df['YearRemodAdd']
    df['HasPool']     = (df['PoolArea'] > 0).astype(int)
    df['HasGarage']   = (df['GarageArea'] > 0).astype(int)
    df['HasBasement'] = (df['TotalBsmtSF'] > 0).astype(int)
    return df

train_fe = engineer_features(train)
test_fe  = engineer_features(test)

X = train_fe.drop(columns=['SalePrice', 'Id'])
y = np.log1p(train['SalePrice'])
X_test = test_fe.drop(columns=['Id'])

num_cols = X.select_dtypes(include=['int64','float64']).columns.tolist()
cat_cols = X.select_dtypes(include=['object']).columns.tolist()

# shared preprocessor
num_tr = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])
cat_tr = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value',
                               unknown_value=-1))
])
preprocessor = ColumnTransformer(transformers=[
    ('num', num_tr, num_cols),
    ('cat', cat_tr, cat_cols)
])

# define 3 models
rf_pipe = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(
        n_estimators=300, max_depth=15,
        max_features='sqrt', min_samples_leaf=2,
        random_state=42, n_jobs=-1))
])

xgb_pipe = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', XGBRegressor(
        learning_rate=0.01, max_depth=5,
        subsample=0.7, colsample_bytree=0.6,
        n_estimators=1000,
        random_state=42, verbosity=0))
])

lgb_pipe = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', lgb.LGBMRegressor(
        n_estimators=2000, learning_rate=0.005,
        num_leaves=63, subsample=0.8,
        colsample_bytree=0.5, min_child_samples=20,
        reg_alpha=0.5, reg_lambda=0.1,
        random_state=42, verbose=-1))
])

# fit all 3 on full training data
print("Training all 3 models...")
rf_pipe.fit(X, y)
xgb_pipe.fit(X, y)
lgb_pipe.fit(X, y)
print("Done!")

# predict on test
rf_preds  = rf_pipe.predict(X_test)
xgb_preds = xgb_pipe.predict(X_test)
lgb_preds = lgb_pipe.predict(X_test)

# simple average
avg_preds = (rf_preds + xgb_preds + lgb_preds) / 3

submission_avg = pd.DataFrame({
    'Id'       : test['Id'],
    'SalePrice': np.expm1(avg_preds)
})
submission_avg.to_csv('ensemble_avg_submission.csv', index=False)
print("Simple average submission saved!")
print(f"Prediction range: ${np.expm1(avg_preds).min():,.0f} to ${np.expm1(avg_preds).max():,.0f}")

Training all 3 models...
Done!
Simple average submission saved!
Prediction range: $55,396 to $496,715


# Weighted averaging (give less weight to RF):

In [2]:
# weighted average — give more weight to better models
# LightGBM best (0.12603), XGBoost second (0.12930), RF worst (0.14864)

weights = {
    'lgb': 0.5,   # best model — highest weight
    'xgb': 0.35,  # second best
    'rf' : 0.15   # worst — lowest weight
}

weighted_preds = (
    weights['lgb'] * lgb_preds +
    weights['xgb'] * xgb_preds +
    weights['rf']  * rf_preds
)

submission_weighted = pd.DataFrame({
    'Id'       : test['Id'],
    'SalePrice': np.expm1(weighted_preds)
})
submission_weighted.to_csv('ensemble_weighted_submission.csv', index=False)
print("Weighted average submission saved!")

# also try LightGBM + XGBoost only (no RF)
lgb_xgb_preds = 0.6 * lgb_preds + 0.4 * xgb_preds

submission_lgb_xgb = pd.DataFrame({
    'Id'       : test['Id'],
    'SalePrice': np.expm1(lgb_xgb_preds)
})
submission_lgb_xgb.to_csv('ensemble_lgb_xgb_submission.csv', index=False)
print("LGB+XGB submission saved!")

Weighted average submission saved!
LGB+XGB submission saved!


# Tune the LGB/XGB blend ratio:

In [3]:
# try different weight ratios
ratios = [
    (0.5, 0.5),   # equal
    (0.6, 0.4),   # current best
    (0.7, 0.3),   # more LGB
    (0.4, 0.6),   # more XGB
    (0.8, 0.2),   # heavy LGB
]

print("Weight ratio experiments (LGB / XGB):")
print("Saving submissions for each...")

for lgb_w, xgb_w in ratios:
    preds = lgb_w * lgb_preds + xgb_w * xgb_preds
    sub = pd.DataFrame({
        'Id'       : test['Id'],
        'SalePrice': np.expm1(preds)
    })
    fname = f'blend_lgb{int(lgb_w*10)}_xgb{int(xgb_w*10)}.csv'
    sub.to_csv(fname, index=False)
    print(f"  LGB:{lgb_w} XGB:{xgb_w} → saved {fname}")

print("\nSubmit each to Kaggle and find best ratio!")

Weight ratio experiments (LGB / XGB):
Saving submissions for each...
  LGB:0.5 XGB:0.5 → saved blend_lgb5_xgb5.csv
  LGB:0.6 XGB:0.4 → saved blend_lgb6_xgb4.csv
  LGB:0.7 XGB:0.3 → saved blend_lgb7_xgb3.csv
  LGB:0.4 XGB:0.6 → saved blend_lgb4_xgb6.csv
  LGB:0.8 XGB:0.2 → saved blend_lgb8_xgb2.csv

Submit each to Kaggle and find best ratio!


# Stacking with sklearn:

In [4]:
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score, KFold

# stacking — RF + XGB as base models, Ridge as meta-learner
# note: stacking works on raw models not pipelines here
# preprocess first then stack

# preprocess data
X_t = preprocessor.fit_transform(X)
X_test_t = preprocessor.transform(X_test)

# base models
estimators = [
    ('xgb', XGBRegressor(
        learning_rate=0.01, max_depth=5,
        subsample=0.7, colsample_bytree=0.6,
        n_estimators=1000, random_state=42, verbosity=0)),
    ('lgb', lgb.LGBMRegressor(
        n_estimators=2000, learning_rate=0.005,
        num_leaves=63, subsample=0.8,
        colsample_bytree=0.5, min_child_samples=20,
        reg_alpha=0.5, reg_lambda=0.1,
        random_state=42, verbose=-1))
]

# meta learner — Ridge regression
stacking = StackingRegressor(
    estimators=estimators,
    final_estimator=Ridge(alpha=1.0),
    cv=5,
    n_jobs=-1
)

# evaluate
kfold = KFold(n_splits=5, shuffle=True, random_state=42)
scores_stack = cross_val_score(stacking, X_t, y,
                                cv=kfold,
                                scoring='neg_root_mean_squared_error')

print("=" * 50)
print("   STACKING RESULTS")
print("=" * 50)
print(f"  Stacking RMSE  : {-scores_stack.mean():.4f} ± {scores_stack.std():.4f}")
print(f"  Blend (60/40)  : ~0.1259 (Kaggle)")
print(f"  LightGBM alone : 0.1274 (CV)")
print("=" * 50)

   STACKING RESULTS
  Stacking RMSE  : 0.1256 ± 0.0197
  Blend (60/40)  : ~0.1259 (Kaggle)
  LightGBM alone : 0.1274 (CV)


# Submit stacking to Kaggle:

In [5]:
# fit stacking on full data
stacking.fit(X_t, y)

# predict on test
stack_preds = stacking.predict(X_test_t)
predictions = np.expm1(stack_preds)

submission_stack = pd.DataFrame({
    'Id'       : test['Id'],
    'SalePrice': predictions
})
submission_stack.to_csv('stacking_submission.csv', index=False)
print("Stacking submission saved!")
print(f"Prediction range: ${predictions.min():,.0f} to ${predictions.max():,.0f}")

# check what weights Ridge learned
ridge_meta = stacking.final_estimator_
print(f"\nMeta-model (Ridge) learned weights:")
print(f"  XGBoost weight : {ridge_meta.coef_[0]:.4f}")
print(f"  LightGBM weight: {ridge_meta.coef_[1]:.4f}")

Stacking submission saved!
Prediction range: $47,923 to $537,566

Meta-model (Ridge) learned weights:
  XGBoost weight : 0.7168
  LightGBM weight: 0.3064
